# DAY 11 -- Functional Programming (Lambda Map)

#### Key principles to reinforce :

* **Immutability** → safer, predictable code
* **Lazy evaluation** → efficiency at scale
* **No side effects** → easier reasoning
* **Functions as data** → composability

#### Mental Model - How It All Fits Together

We should think of functional tools as **data transformers**, not containers:

```
DATA
 ↓
map     → transforms values
filter  → removes values
reduce  → collapses values
zip     → aligns values
any/all → short-circuit checks
partial → configures behavior
```


### MC11.1 : The Anonymous Function


> Goal : Write a function `add(x, y)` using `lambda`.


In [1]:
# Write `add(x, y)` using lambda

add = lambda x, y: x + y

print(add(3, 4))

7


> **Deep Dive:** `lambda` creates a function object on the heap but assigns it no name (unless you bind it to a variable). It is purely syntactic sugar for single-expression functions.


---


### MC11.2 : The Mapper


> Goal : Square a list of numbers using `map()`.


In [14]:
# Square a list using map()

nums = [1, 2, 3, 4]

squares = map(lambda x: x * x, nums)
squares_list = list(squares)
print(f"{type(squares)} {squares=}\n{type(squares_list)} {squares_list=}")

<class 'map'> squares=<map object at 0x0000025FD298C5B0>
<class 'list'> squares_list=[1, 4, 9, 16]


> **Deep Dive:** `map(func, list)` pushes the loop into C-speed. It returns an iterator (lazy), not a list. You must cast it with `list()` to trigger execution.


---


### MC11.3 : The Filter


> Goal : Remove all negative numbers from a list using `filter()`.


In [15]:
# Remove negative numbers using `filter()`
nums = [-3, -1, 0, 2, 5]

positives = filter(lambda x: x >= 0, nums)
print(list(positives))

[0, 2, 5]


In [4]:
## Extra: `filter(None, iterable)` removes all falsy values
mixed = [0, 1, "", "hi", None, 5]
print(list(filter(None, mixed)))

[1, 'hi', 5]


> **Deep Dive:** `filter(func, list)` keeps items where `func(item)` returns Truthy. Passing `None` as the function automatically filters out Falsy values (`0`, `""`, `None`).


---


### MC11.4 : The Reducer


> Goal : Calculate the product of a list (`1 * 2 * 3 * 4`) using `functools.reduce`.


In [5]:
# Compute product of list using reduce
from functools import reduce

nums = [1, 2, 3, 4]

product = reduce(lambda a, b: a * b, nums)
print(product)

24


> **Deep Dive:** Reduce collapses a list into a single value. It takes item 1 and 2, applies the function, takes the result and item 3, applies again… until one item remains.


---


### MC11.5 : The Custom Sort Key


> Goal : Sort `["100px", "20px", "3px"]` numerically (so `3px` comes first).


In [16]:
# Sort pixel values numerically
data = ["100px", "20px", "3px"]

sorted_data = sorted(data, key=lambda x: int(x[:-2])) ## remove 'px' (last 2 chars) , convert to int
print(sorted_data) ## aside from the ordering, the values remain unchanged -- strings with 'px'

['3px', '20px', '100px']


> **Deep Dive:** `sorted(data, key=lambda x: int(x[:-2]))`.
> The key function transforms the item **only for comparison purposes**, leaving the original data intact.


---


### MC11.6 : The Zip Lock


> Goal : Combine `names = ["A", "B"]` and `ages = [20, 30]` into a dictionary.


In [7]:
# Combine names and ages into a dictionary
names = ["A", "B"]
ages = [20, 30]

user_map = dict(zip(names, ages))
print(user_map)

{'A': 20, 'B': 30}


In [8]:
## Alternative using dictionary comprehension
{n: a for n, a in zip(names, ages)}

{'A': 20, 'B': 30}

> **Deep Dive:** `dict(zip(names, ages))`.
> `zip` creates an iterator of tuples. `dict()` consumes those tuples to build the hash map.


---


### MC11.7 : List Comprehension Speed


> Goal : Compare `map(lambda...)` vs `[x ... for x ...]`.


In [21]:
# Compare map vs list comprehension

import time

for s in [1000, 1_000_000, 10_000_000]:
    nums = list(range(s))

    start = time.time()
    [x * x for x in nums]
    lc_time = time.time() - start

    start = time.time()
    list(map(lambda x: x * x, nums))
    m_time = time.time() - start
    
    print(f"For a list of size {s:>11,} --> List comprehension : {lc_time:.6f}s | `map()` : {m_time:.6f}s")


For a list of size       1,000 --> List comprehension : 0.000000s | `map()` : 0.000000s
For a list of size   1,000,000 --> List comprehension : 0.251846s | `map()` : 0.392780s
For a list of size  10,000,000 --> List comprehension : 2.386519s | `map()` : 3.980995s


> **Deep Dive:** List comprehensions are generally faster than `map + lambda` because they avoid the overhead of calling a Python function frame for every single item.


---


### MC11.8 : Any / All


> Goal : Check if **any** number in a list is negative. Check if **all** are positive.


In [10]:
# Check any negative; check all positive
all_pos = [1, 2, 3, 4]
all_neg = [-5, -2, -3]
pos_neg = [1, -2, 3]

print(any(x < 0 for x in all_pos))   ## stops at first True (first negative)
print(all(x > 0 for x in all_pos))   ## stops at first False (first non-positive)
print(any(x < 0 for x in all_neg))   ## checks all, returns True
print(all(x > 0 for x in all_neg))   ## stops at first False (first non-positive)
print(any(x < 0 for x in pos_neg))   ## stops at first True (first negative)
print(all(x > 0 for x in pos_neg))   ## stops at first False (first non-positive)

False
True
True
False
True
False


> **Deep Dive:** These are short-circuiting operators.
> * `any()` stops at the first `True`.
> * `all()` stops at the first `False`.
> This is **O(1)** in the best case.


---


### MC11.9 : Partial Functions


> Goal : Create a function `power(base, exp)`. Use `functools.partial` to create a new function `square(x)` that locks `exp = 2`.


In [11]:
# Freeze arguments using `functools.partial`
from functools import partial

def power(base, exp):
    return base ** exp

square = partial(power, exp=2)

print(square(5))
print(square(10))

25
100


> **Deep Dive:** Partials “freeze” arguments. This is useful when you need to pass a function to a callback (like in UI frameworks) that expects fewer arguments.


---


### MC11.10 : The Immutability Test


> Goal : Try to modify a tuple inside a `map` function.


In [32]:
# Try to mutate a tuple inside map()
data = [(1, 2), (3, 4)]

def attempt_mutation(t):
    # t[0] = 99             ## !! would raise TypeError !!
    return (t[0] * 10, t[1])

result = list(map(attempt_mutation, data))
print(f"After mutation attempt: {result=}\nOriginal data: \t\t  {data=}") ## original data unchanged

After mutation attempt: result=[(10, 2), (30, 4)]
Original data: 		  data=[(1, 2), (3, 4)]


> **Deep Dive:** Functional programming relies on **immutability**. Functions should not have side effects (modifying global state). They should receive input and return new output.


---
